# Multimodal Vision-Language (VLM) Lab - Garmin Hypnogram & Biometric Chart Analysis

## 1. Overview & Objective
Modern health intelligence requires multimodal comprehension. While tabular time series describe macro metrics, visual hypnograms capture nuanced temporal microstructure:
- **Sleep Architecture Cycles**: The normal 90-120 minute progression from Light sleep (N1/N2) to Deep sleep (N3) and REM.
- **Nocturnal Stress & Micro-Arousals**: Sharp upward spikes during deep sleep indicate sympathetic nervous system surges.
- **VLM Chart Interpretation**: Transforming visual plots into structured clinical observations via Vision-Language Models (VLMs).

In [ ]:
# Setup imports
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use(
    "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default"
)
print("Visualization libraries initialized.")

## 2. Hypnogram Generator & Step-Chart Rendering
We synthesize a continuous 8-hour sleep session with standard physiological sleep stage transitions:
- Level 0: Deep Sleep (N3, restorative)
- Level 1: Light Sleep (N1/N2)
- Level 2: REM Sleep (dreaming, mental restoration)
- Level 3: Awake / Micro-arousal

In [ ]:
# Generate 8 hours of sleep epochs (30-second intervals = 960 epochs)
np.random.seed(101)
start_time = pd.Timestamp("2026-09-18 23:00:00")
timestamps = [start_time + pd.Timedelta(seconds=30 * i) for i in range(960)]

# Simulate realistic physiological hypnogram cycle
# Early night: heavy deep sleep; Late night: predominantly REM & Light
stages = []
for i in range(960):
    progress = i / 960
    if i < 20:  # Falling asleep
        stage = 3 if i < 10 else 1
    elif progress < 0.4:  # First half of night: Deep sleep concentrated
        probs = [0.45, 0.40, 0.12, 0.03]
        stage = np.random.choice([0, 1, 2, 3], p=probs)
    else:  # Second half: REM and Light sleep dominate
        probs = [0.08, 0.50, 0.38, 0.04]
        stage = np.random.choice([0, 1, 2, 3], p=probs)
    stages.append(stage)

df_hypno = pd.DataFrame({"timestamp": timestamps, "stage": stages})

# Render clinical Garmin-style Hypnogram
fig, ax = plt.subplots(figsize=(14, 5))

stage_labels = {0: "Deep Sleep", 1: "Light Sleep", 2: "REM Sleep", 3: "Awake"}
colors = {0: "#1d3557", 1: "#457b9d", 2: "#a8dadc", 3: "#e63946"}

# Step chart
ax.step(df_hypno["timestamp"], df_hypno["stage"], where="post", color="#2b2d42", linewidth=1.5)

# Shaded zones for clarity
for s, label in stage_labels.items():
    ax.axhspan(s - 0.4, s + 0.4, color=colors[s], alpha=0.25, label=label)

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(["Deep (N3)", "Light (N1/N2)", "REM", "Awake"], fontsize=11, fontweight="bold")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_title(
    "Garmin Visual Hypnogram: 8-Hour Nocturnal Sleep Architecture", fontsize=14, fontweight="bold"
)
ax.set_xlabel("Time of Night (Local)")
ax.set_ylim(-0.5, 3.5)
plt.tight_layout()

# Save rendered chart for VLM ingestion
CHART_DIR = Path("../data/processed/charts")
if not CHART_DIR.exists():
    CHART_DIR = Path("data/processed/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)
hypno_img_path = CHART_DIR / "sample_hypnogram.png"
plt.savefig(hypno_img_path, dpi=150)
plt.show()
print(f"✅ Hypnogram chart saved to: {hypno_img_path.resolve()}")

## 3. Multimodal VLM Prompt Construction
We formulate structured prompt instructions to feed this hypnogram into a Vision-Language Model (such as GPT-4o, Claude 3.5 Sonnet, or Qwen2-VL) to extract clinical and athletic insights.

In [ ]:
vlm_prompt = """You are an expert sports physiologist and sleep medicine physician analyzing a patient's Garmin hypnogram.
Examine the attached hypnogram image and evaluate:
1. Sleep Architecture Distribution: Are slow-wave deep sleep cycles concentrated properly in the first third of the night?
2. Sleep Fragmentation: Count the frequency of micro-arousal spikes into the 'Awake' tier.
3. REM Density: Does REM sleep duration expand normally towards the morning hours?
4. Recovery Recommendation: Provide actionable recommendations regarding sleep hygiene, meal timing, and training readiness.

Output your findings in structured JSON format with fields: ['deep_sleep_assessment', 'rem_assessment', 'fragmentation_severity', 'actionable_advice'].
"""

print("Constructed VLM Prompt Payload:")
print("-" * 60)
print(vlm_prompt)
print("-" * 60)

## 4. Summary & Vision System Roadmap

### Key Findings
- **Visual Granularity**: The rendered hypnogram step-chart accurately captures the physiological ultradian sleep cycle, making sleep quality interpretable both for humans and VLMs.
- **Multimodal Complement**: Visual chart parsing provides qualitative context that scalar summary metrics (such as a single `sleep_score: 75`) fail to communicate (e.g. early waking vs fragmented middle sleep).

### Next Steps
- Integrate with `src/vision/chart_renderer.py` and `src/vision/vlm_analyzer.py` to automatically render hypnograms during the daily cron and pass them to multimodal LLM agents.